In [4]:
from config import NAVY, RED, GREY, BG

In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///data/creditrisk.db')
df = pd.read_sql('SELECT * FROM german_credit', engine)

# Convert target to binary: 0 = Good, 1 = Bad
df['target'] = df['target'].map({1: 0, 2: 1})

print(df['target'].value_counts())

target
0    700
1    300
Name: count, dtype: int64


In [2]:
mappings = {
    'checking_account': {'A11': '<0 DM', 'A12': '0-200 DM', 'A13': '>200 DM', 'A14': 'no account'},
    'credit_history': {'A30': 'no credits', 'A31': 'all paid', 'A32': 'existing paid', 'A33': 'delay in past', 'A34': 'critical account'},
    'savings_account': {'A61': '<100 DM', 'A62': '100-500 DM', 'A63': '500-1000 DM', 'A64': '>1000 DM', 'A65': 'no savings'},
    'employment': {'A71': 'unemployed', 'A72': '<1 yr', 'A73': '1-4 yrs', 'A74': '4-7 yrs', 'A75': '>7 yrs'},
    'personal_status': {'A91': 'male divorced', 'A92': 'female divorced', 'A93': 'male single', 'A94': 'male married', 'A95': 'female single'},
    'other_debtors': {'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor'},
    'property': {'A121': 'real estate', 'A122': 'life insurance', 'A123': 'car', 'A124': 'no property'},
    'other_installments': {'A141': 'bank', 'A142': 'stores', 'A143': 'none'},
    'housing': {'A151': 'rent', 'A152': 'own', 'A153': 'free'},
    'job': {'A171': 'unskilled non-resident', 'A172': 'unskilled resident', 'A173': 'skilled', 'A174': 'highly skilled'},
    'telephone': {'A191': 'no', 'A192': 'yes'},
    'foreign_worker': {'A201': 'yes', 'A202': 'no'},
    'purpose': {'A40': 'car new', 'A41': 'car used', 'A42': 'furniture', 'A43': 'radio/tv', 'A44': 'appliances',
                'A45': 'repairs', 'A46': 'education', 'A47': 'vacation', 'A48': 'retraining', 'A49': 'business', 'A410': 'other'}
}

for col, mapping in mappings.items():
    df[col] = df[col].map(mapping)

df.head()

,checking_account,duration,credit_history,purpose,credit_amount,savings_account,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installments,housing,existing_credits,job,dependents,telephone,foreign_worker,target
0,<0 DM,6,critical account,radio/tv,1169,no savings,>7 yrs,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,0
1,0-200 DM,48,existing paid,radio/tv,5951,<100 DM,1-4 yrs,2,female divorced,none,...,real estate,22,none,own,1,skilled,1,no,yes,1
2,no account,12,critical account,education,2096,<100 DM,4-7 yrs,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,no,yes,0
3,<0 DM,42,existing paid,furniture,7882,<100 DM,4-7 yrs,2,male single,guarantor,...,life insurance,45,none,free,1,skilled,2,no,yes,0
4,<0 DM,24,delay in past,car new,4870,<100 DM,1-4 yrs,3,male single,none,...,no property,53,none,free,2,skilled,2,no,yes,1


In [ ]:
import numpy as np

def calculate_woe_iv(df, feature, target):
    total_good = (df[target] == 0).sum()
    total_bad = (df[target] == 1).sum()
    
    stats = df.groupby(feature)[target].agg(
        bad=lambda x: (x == 1).sum(),
        good=lambda x: (x == 0).sum()
    ).reset_index()
    
    stats['bad_rate'] = stats['bad'] / total_bad
    stats['good_rate'] = stats['good'] / total_good
    stats['bad_rate'] = stats['bad_rate'].replace(0, 0.0001)
    stats['good_rate'] = stats['good_rate'].replace(0, 0.0001)
    stats['woe'] = np.log(stats['good_rate'] / stats['bad_rate'])
    stats['iv'] = (stats['good_rate'] - stats['bad_rate']) * stats['woe']
    
    return stats[['woe', 'iv', feature]].set_index(feature)['woe'], stats['iv'].sum()

# Calculate IV for all categorical features
cat_cols = df.select_dtypes(include='str').columns.tolist()
iv_summary = {}

for col in cat_cols:
    _, iv = calculate_woe_iv(df, col, 'target')
    iv_summary[col] = round(iv, 4)

iv_df = pd.DataFrame.from_dict(iv_summary, orient='index', columns=['IV'])
iv_df = iv_df.sort_values('IV', ascending=False)

print("Information Value Summary:")
print(iv_df)

In [ ]:
# Apply WoE encoding to categorical features
df_woe = df.copy()

for col in cat_cols:
    woe_map, _ = calculate_woe_iv(df, col, 'target')
    df_woe[col] = df_woe[col].map(woe_map)

# Save feature store back to SQLite
df_woe.to_sql('feature_store', engine, if_exists='replace', index=False)

print("Feature store saved to SQLite!")
print(df_woe.shape)
df_woe.head()

Feature store saved to SQLite!
(1000, 21)


,checking_account,duration,credit_history,purpose,credit_amount,savings_account,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installments,housing,existing_credits,job,dependents,telephone,foreign_worker,target
0,-0.818099,6,0.733741,0.410063,1169,0.704246,0.235566,4,0.165548,0.000525,...,0.461035,67,0.121179,0.194156,2,0.022780,1,0.098638,-0.034867,0
1,-0.401392,48,-0.088319,0.410063,5951,-0.271358,-0.032103,2,-0.235341,0.000525,...,0.461035,22,0.121179,0.194156,1,0.022780,1,-0.064691,-0.034867,1
2,1.176263,12,0.733741,-0.606136,2096,-0.271358,0.394415,2,0.165548,0.000525,...,0.461035,49,0.121179,0.194156,1,0.097164,2,-0.064691,-0.034867,0
3,-0.818099,42,-0.088319,-0.095557,7882,-0.271358,0.394415,2,0.165548,0.587787,...,-0.028573,45,0.121179,-0.472604,1,0.022780,2,-0.064691,-0.034867,0
4,-0.818099,24,-0.085158,-0.359200,4870,-0.271358,-0.032103,3,0.165548,0.000525,...,-0.586082,53,0.121179,-0.472604,2,0.022780,2,-0.064691,-0.034867,1


In [ ]:
# Drop weak features (IV < 0.02)
cols_to_drop = ['job', 'telephone']
df_model = df_woe.drop(columns=cols_to_drop)

# Save final model-ready dataset to SQLite
df_model.to_sql('model_ready', engine, if_exists='replace', index=False)

print(f"Dropped: {cols_to_drop}")
print(f"Final shape: {df_model.shape}")
print("Model-ready dataset saved to SQLite ✅")

Dropped: ['job', 'telephone']
Final shape: (1000, 19)
Model-ready dataset saved to SQLite ✅
